# Individual Task 1: Part 1 - Section 3 (Data Analysis)
Customer churn / propensity modelling across two independent data sources.
 
Dataset A: Telco Customer Churn      (Kaggle / IBM sample data)
Dataset B: Online Retail II          (UCI Machine Learning Repository)
 
Two models are fitted to each dataset:
    - Logistic Regression  (interpretable baseline; coefficients -> odds ratios)
    - Gradient Boosting    (higher-capacity comparator)
 
Expected files in ./data/ :
    WA_Fn-UseC_-Telco-Customer-Churn.csv
    online_retail_II.xlsx
 
Run:  python churn_analysis.py

In [4]:
#importing libraries 
import pandas as pd
import numpy as np
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score)

#### Loading and cleaning Telco data

In [7]:
telco = pd.read_csv("./WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Rows and columns:", telco.shape)
 
# TotalCharges is stored as text and 11 rows are blank.
# Those 11 customers all have tenure = 0, so they never got a bill. Drop them.
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")
print("Blank TotalCharges:", telco["TotalCharges"].isna().sum())
telco = telco.dropna(subset=["TotalCharges"])
 
# customerID is just an ID, it has no predictive value
telco = telco.drop(columns=["customerID"])
 
# Turn the target into 1 = churned, 0 = stayed
telco["Churn"] = (telco["Churn"] == "Yes").astype(int)
 
print("Churn rate:", round(telco["Churn"].mean(), 3))
telco.head()

Rows and columns: (7043, 21)
Blank TotalCharges: 11
Churn rate: 0.266


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


#### Preparing Telco features

In [8]:
y_telco = telco["Churn"]
X_telco = telco.drop(columns=["Churn"])
 
X_telco = pd.get_dummies(X_telco, drop_first=True)
print("Columns after encoding:", X_telco.shape[1])
 
X_train, X_test, y_train, y_test = train_test_split(
    X_telco, y_telco, test_size=0.25, stratify=y_telco, random_state=42
)
 
# Logistic regression works better when numbers are on the same scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Columns after encoding: 30


#### Logistic Regression

In [9]:
# class_weight="balanced" tells the model to take the smaller churn group as seriously as the larger one
logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
logreg.fit(X_train_scaled, y_train)
 
pred_lr = logreg.predict(X_test_scaled)
prob_lr = logreg.predict_proba(X_test_scaled)[:, 1]
 
print("TELCO - LOGISTIC REGRESSION")
print(classification_report(y_test, pred_lr, digits=3))
print("Confusion matrix:")
print(confusion_matrix(y_test, pred_lr))
print("ROC-AUC:", round(roc_auc_score(y_test, prob_lr), 3))
print("PR-AUC:", round(average_precision_score(y_test, prob_lr), 3))

TELCO - LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0      0.906     0.710     0.796      1291
           1      0.499     0.797     0.613       467

    accuracy                          0.733      1758
   macro avg      0.702     0.753     0.705      1758
weighted avg      0.798     0.733     0.748      1758

Confusion matrix:
[[917 374]
 [ 95 372]]
ROC-AUC: 0.84
PR-AUC: 0.623


#### Gradient Boosting 

In [11]:
gb = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
gb.fit(X_train, y_train)         
 
pred_gb = gb.predict(X_test)
prob_gb = gb.predict_proba(X_test)[:, 1]
 
print("TELCO - GRADIENT BOOSTING")
print(classification_report(y_test, pred_gb, digits=3))
print("Confusion matrix:")
print(confusion_matrix(y_test, pred_gb))
print("ROC-AUC:", round(roc_auc_score(y_test, prob_gb), 3))
print("PR-AUC:", round(average_precision_score(y_test, prob_gb), 3))

TELCO - GRADIENT BOOSTING
              precision    recall  f1-score   support

           0      0.896     0.757     0.821      1291
           1      0.530     0.758     0.624       467

    accuracy                          0.757      1758
   macro avg      0.713     0.757     0.722      1758
weighted avg      0.799     0.757     0.768      1758

Confusion matrix:
[[977 314]
 [113 354]]
ROC-AUC: 0.832
PR-AUC: 0.642


In [12]:
# A positive coefficient means that feature pushes churn UP
coefficients = pd.DataFrame({
    "feature": X_telco.columns,
    "coefficient": logreg.coef_[0]
})
coefficients["size"] = coefficients["coefficient"].abs()
coefficients = coefficients.sort_values("size", ascending=False)
 
print("Top 10 churn drivers (Telco):")
print(coefficients.head(10)[["feature", "coefficient"]])

Top 10 churn drivers (Telco):
                           feature  coefficient
1                           tenure    -1.233570
2                   MonthlyCharges    -1.024866
10     InternetService_Fiber optic     0.798656
25               Contract_Two year    -0.625004
3                     TotalCharges     0.598146
24               Contract_One year    -0.296701
21                 StreamingTV_Yes     0.280310
23             StreamingMovies_Yes     0.270124
9                MultipleLines_Yes     0.206064
28  PaymentMethod_Electronic check     0.178524


#### Loading and cleaning Retail data

In [14]:
sheet1 = pd.read_excel("./online_retail_II.xlsx", sheet_name="Year 2009-2010")
sheet2 = pd.read_excel("./online_retail_II.xlsx", sheet_name="Year 2010-2011")
retail = pd.concat([sheet1, sheet2], ignore_index=True)
print("Raw rows:", len(retail))
 
retail.columns = ["Invoice", "StockCode", "Description", "Quantity",
                  "InvoiceDate", "Price", "CustomerID", "Country"]
 
retail["is_cancelled"] = retail["Invoice"].astype(str).str.startswith("C")
 
# Guest checkouts have no CustomerID, so we cannot track them
retail = retail.dropna(subset=["CustomerID"])
retail = retail[retail["Price"] > 0]
retail = retail[retail["Quantity"] > 0]
 
retail["CustomerID"] = retail["CustomerID"].astype(int)
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"])
retail["line_total"] = retail["Quantity"] * retail["Price"]
 
print("Rows after cleaning:", len(retail))

Raw rows: 1067371
Rows after cleaning: 805549


#### Building the churn label for Retail

In [15]:
# There is no churn column, so we make one.
# Everything BEFORE the cutoff becomes our features.
# Everything AFTER tells us whether the customer came back.
cutoff = pd.Timestamp("2011-06-09")
 
before = retail[retail["InvoiceDate"] < cutoff]
after = retail[retail["InvoiceDate"] >= cutoff]
 
purchases = before[before["is_cancelled"] == False]
snapshot = before["InvoiceDate"].max()
 
# Building one row per customer
customers = pd.DataFrame()
customers["last_purchase"] = purchases.groupby("CustomerID")["InvoiceDate"].max()
customers["first_purchase"] = purchases.groupby("CustomerID")["InvoiceDate"].min()
customers["frequency"] = purchases.groupby("CustomerID")["Invoice"].nunique()
customers["monetary"] = purchases.groupby("CustomerID")["line_total"].sum()
customers["distinct_products"] = purchases.groupby("CustomerID")["StockCode"].nunique()
customers["total_units"] = purchases.groupby("CustomerID")["Quantity"].sum()
 
# Recency = days since last purchase. Tenure = days as a customer.
customers["recency_days"] = (snapshot - customers["last_purchase"]).dt.days
customers["tenure_days"] = (snapshot - customers["first_purchase"]).dt.days
customers["avg_order_value"] = customers["monetary"] / customers["frequency"]
 
customers = customers.drop(columns=["last_purchase", "first_purchase"])
 
# Did they come back after the cutoff?
came_back = after["CustomerID"].unique()
customers["churn"] = (~customers.index.isin(came_back)).astype(int)
 
print("Customers:", len(customers))
print("Churn rate:", round(customers["churn"].mean(), 3))
customers.head()

Customers: 4966
Churn rate: 0.48


,frequency,monetary,distinct_products,total_units,recency_days,tenure_days,avg_order_value,churn
CustomerID,,,,,,,,
12346,12,77556.46,27,74285,141,541,6463.038333,1
12347,4,3146.75,90,1945,62,220,786.687500,0
12348,4,1709.40,25,2497,64,254,427.350000,0
12349,3,2671.14,90,993,223,405,890.380000,0
12350,1,334.40,17,197,126,126,334.400000,1


#### Applying both the models in Retail

In [16]:
y_retail = customers["churn"]
X_retail = customers.drop(columns=["churn"])
 
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_retail, y_retail, test_size=0.25, stratify=y_retail, random_state=42
)
 
scaler_r = StandardScaler()
Xr_train_scaled = scaler_r.fit_transform(Xr_train)
Xr_test_scaled = scaler_r.transform(Xr_test)
 
logreg_r = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
logreg_r.fit(Xr_train_scaled, yr_train)
pred_lr_r = logreg_r.predict(Xr_test_scaled)
prob_lr_r = logreg_r.predict_proba(Xr_test_scaled)[:, 1]
 
print("RETAIL - LOGISTIC REGRESSION")
print(classification_report(yr_test, pred_lr_r, digits=3))
print("ROC-AUC:", round(roc_auc_score(yr_test, prob_lr_r), 3))
print("PR-AUC:", round(average_precision_score(yr_test, prob_lr_r), 3))
 
gb_r = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
gb_r.fit(Xr_train, yr_train)
pred_gb_r = gb_r.predict(Xr_test)
prob_gb_r = gb_r.predict_proba(Xr_test)[:, 1]
 
print("\nRETAIL - GRADIENT BOOSTING")
print(classification_report(yr_test, pred_gb_r, digits=3))
print("ROC-AUC:", round(roc_auc_score(yr_test, prob_gb_r), 3))
print("PR-AUC:", round(average_precision_score(yr_test, prob_gb_r), 3))

RETAIL - LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0      0.764     0.712     0.737       646
           1      0.709     0.762     0.735       596

    accuracy                          0.736      1242
   macro avg      0.737     0.737     0.736      1242
weighted avg      0.738     0.736     0.736      1242

ROC-AUC: 0.81
PR-AUC: 0.775

RETAIL - GRADIENT BOOSTING
              precision    recall  f1-score   support

           0      0.742     0.669     0.704       646
           1      0.676     0.748     0.710       596

    accuracy                          0.707      1242
   macro avg      0.709     0.709     0.707      1242
weighted avg      0.710     0.707     0.707      1242

ROC-AUC: 0.787
PR-AUC: 0.744


In [17]:
coefficients_r = pd.DataFrame({
    "feature": X_retail.columns,
    "coefficient": logreg_r.coef_[0]
})
coefficients_r["size"] = coefficients_r["coefficient"].abs()
coefficients_r = coefficients_r.sort_values("size", ascending=False)
 
print("Churn drivers (Retail):")
print(coefficients_r[["feature", "coefficient"]])

Churn drivers (Retail):
             feature  coefficient
4       recency_days     0.909220
0          frequency    -0.617527
2  distinct_products    -0.566406
5        tenure_days    -0.167029
3        total_units     0.053014
6    avg_order_value    -0.037449
1           monetary     0.002680


#### Summary Table

In [18]:
def get_scores(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": round((y_true == y_pred).mean(), 3),
        "Precision": round(classification_report(
            y_true, y_pred, output_dict=True)["1"]["precision"], 3),
        "Recall": round(classification_report(
            y_true, y_pred, output_dict=True)["1"]["recall"], 3),
        "F1": round(classification_report(
            y_true, y_pred, output_dict=True)["1"]["f1-score"], 3),
        "ROC-AUC": round(roc_auc_score(y_true, y_prob), 3),
        "PR-AUC": round(average_precision_score(y_true, y_prob), 3),
    }
 
summary = pd.DataFrame([
    get_scores("Telco - Logistic Regression", y_test, pred_lr, prob_lr),
    get_scores("Telco - Gradient Boosting", y_test, pred_gb, prob_gb),
    get_scores("Retail - Logistic Regression", yr_test, pred_lr_r, prob_lr_r),
    get_scores("Retail - Gradient Boosting", yr_test, pred_gb_r, prob_gb_r),
])
 
print(summary.to_string(index=False))
summary.to_csv("model_comparison.csv", index=False)

                       Model  Accuracy  Precision  Recall    F1  ROC-AUC  PR-AUC
 Telco - Logistic Regression     0.733      0.499   0.797 0.613    0.840   0.623
   Telco - Gradient Boosting     0.757      0.530   0.758 0.624    0.832   0.642
Retail - Logistic Regression     0.736      0.709   0.762 0.735    0.810   0.775
  Retail - Gradient Boosting     0.707      0.676   0.748 0.710    0.787   0.744
